# Таргетированный маркетинг
Имеется 2 файла с данными - `train.csv` и `test.csv` для тренировки и тестирования моделей соответственно.

В файле `train.csv` есть поля `'AcceptedCmp1'`, `'AcceptedCmp2'`, `'AcceptedCmp3'`, `'AcceptedCmp4'`, `'AcceptedCmp5'`, `'Response'`, а в `test.csv` нет.

Эти поля содержат бинарный признак - `0/1`, принял человек предложение в конкретной маркетинговой кампании или нет.

Необходимо предсказать бинарное поле `'Target'` для данных в файле `test.csv`, где `1` соответствует тому, что человек в принципе примет хотя бы одно предложение из всех рекламных кампаний, а `0 соответсвует тому, что во всех кампаниях будет отказ.

Данная модель поможет бизнесу предсказывать целесообразность проведения рекламных кампаний для каждого человека с учётом всех известных о нём данных, что снизит издержки и повысит прибыль.

Ответ должен иметь размерность 449 строк на 2 столбца с полями `'ID'` и `'Target'`.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.datasets import fetch_openml

import warnings
warnings.filterwarnings('ignore')

Загрузим датасеты `train.csv` и `test.csv` и посмотрим на них

In [2]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [3]:
train.head()

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,04/09/12,58,635,...,7,0,0,0,0,0,0,3,11,1
1,6182,1984,Graduation,Together,26646.0,1,0,10/02/14,26,11,...,6,0,0,0,0,0,0,3,11,0
2,5324,1981,PhD,Married,58293.0,1,0,19/01/14,94,173,...,5,0,0,0,0,0,0,3,11,0
3,7446,1967,Master,Together,62513.0,0,1,09/09/13,16,520,...,6,0,0,0,0,0,0,3,11,0
4,965,1971,Graduation,Divorced,55635.0,0,1,13/11/12,34,235,...,6,0,0,0,0,0,0,3,11,0


In [4]:
train.describe(include='all')

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
count,1792.000000,1792.000000,1792,1792,1773.000000,1792.000000,1792.000000,1792,1792.000000,1792.000000,...,1792.000000,1792.000000,1792.000000,1792.000000,1792.000000,1792.000000,1792.000000,1792.0,1792.0,1792.000000
unique,NaN,NaN,5,8,NaN,NaN,NaN,636,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,NaN,Graduation,Married,NaN,NaN,NaN,31/08/12,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,NaN,899,702,NaN,NaN,NaN,10,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,5564.762277,1968.880022,NaN,NaN,52729.121263,0.433594,0.509487,NaN,48.957031,310.848772,...,5.293527,0.066964,0.073103,0.071987,0.065848,0.013393,0.007812,3.0,11.0,0.148438
std,3232.974510,11.996584,NaN,NaN,25943.155423,0.538883,0.542879,NaN,29.007399,339.840162,...,2.387807,0.250030,0.260378,0.258538,0.248086,0.114982,0.088067,0.0,0.0,0.355632
min,0.000000,1893.000000,NaN,NaN,1730.000000,0.000000,0.000000,NaN,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.0,11.0,0.000000
25%,2806.250000,1959.000000,NaN,NaN,35790.000000,0.000000,0.000000,NaN,24.000000,25.000000,...,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.0,11.0,0.000000
50%,5453.500000,1970.000000,NaN,NaN,52203.000000,0.000000,0.000000,NaN,49.000000,183.500000,...,6.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.0,11.0,0.000000
75%,8372.250000,1977.000000,NaN,NaN,69084.000000,1.000000,1.000000,NaN,74.000000,513.250000,...,7.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.0,11.0,0.000000


In [5]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 1792 entries, 0 to 1791
Data columns (total 29 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   1792 non-null   int64  
 1   Year_Birth           1792 non-null   int64  
 2   Education            1792 non-null   str    
 3   Marital_Status       1792 non-null   str    
 4   Income               1773 non-null   float64
 5   Kidhome              1792 non-null   int64  
 6   Teenhome             1792 non-null   int64  
 7   Dt_Customer          1792 non-null   str    
 8   Recency              1792 non-null   int64  
 9   MntWines             1792 non-null   int64  
 10  MntFruits            1792 non-null   int64  
 11  MntMeatProducts      1792 non-null   int64  
 12  MntFishProducts      1792 non-null   int64  
 13  MntSweetProducts     1792 non-null   int64  
 14  MntGoldProds         1792 non-null   int64  
 15  NumDealsPurchases    1792 non-null   int64  
 16 

Добавим столбец `Target` как итоговый `OR` для полей `'AcceptedCmp1'`, `'AcceptedCmp2'`, `'AcceptedCmp3'`, `'AcceptedCmp4'`, `'AcceptedCmp5'`, `'Response'`

In [6]:
train['Target'] = (train['AcceptedCmp1'] + train['AcceptedCmp2'] +
                   train['AcceptedCmp3'] + train['AcceptedCmp4'] + 
                   train['AcceptedCmp5'] + train['Response']).apply(lambda x: 1 if x > 0 else 0)

In [7]:
test.head()

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,MntSweetProducts,MntGoldProds,NumDealsPurchases,NumWebPurchases,NumCatalogPurchases,NumStorePurchases,NumWebVisitsMonth,Complain,Z_CostContact,Z_Revenue
0,286,1952,Graduation,Single,44213.0,1,1,29/11/13,48,95,...,4,7,4,2,1,5,6,0,3,11
1,6374,1954,PhD,Married,36930.0,0,1,17/05/13,50,223,...,2,39,5,5,2,4,8,0,3,11
2,1160,1970,Graduation,Married,13260.0,1,1,23/08/13,48,9,...,2,7,4,3,0,3,8,0,3,11
3,701,1971,Graduation,Married,73691.0,0,1,06/11/13,58,707,...,43,73,2,6,2,8,2,0,3,11
4,4084,1975,Graduation,Together,60934.0,0,1,17/01/14,41,224,...,93,54,2,6,4,11,4,0,3,11


In [8]:
test.info()

<class 'pandas.DataFrame'>
RangeIndex: 448 entries, 0 to 447
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   448 non-null    int64  
 1   Year_Birth           448 non-null    int64  
 2   Education            448 non-null    str    
 3   Marital_Status       448 non-null    str    
 4   Income               443 non-null    float64
 5   Kidhome              448 non-null    int64  
 6   Teenhome             448 non-null    int64  
 7   Dt_Customer          448 non-null    str    
 8   Recency              448 non-null    int64  
 9   MntWines             448 non-null    int64  
 10  MntFruits            448 non-null    int64  
 11  MntMeatProducts      448 non-null    int64  
 12  MntFishProducts      448 non-null    int64  
 13  MntSweetProducts     448 non-null    int64  
 14  MntGoldProds         448 non-null    int64  
 15  NumDealsPurchases    448 non-null    int64  
 16  N

Подготовим датасет

In [9]:
train['Marital_Status'].unique()

<StringArray>
['Single', 'Together', 'Married', 'Divorced', 'Widow', 'Alone', 'Absurd',
 'YOLO']
Length: 8, dtype: str

In [10]:
def prepare (dataset, key='train'):
    education_map = {
        'Basic': 0.1,
        '2n Cycle': 0.3,
        'Master': 0.5,
        'Graduation': 0.7,
        'PhD': 0.9      
    }
    
    dataset['Education'] = dataset['Education'].map(education_map)

    ms_map = {
        'Single': 0.0,
        'YOLO': 0.0,
        'Together': 0.1,
        'Married': 0.2,
        'Divorced': 0.3,
        'Widow': 0.4,
        'Alone': 0.5,
        'Absurd': 0.0   
    }   

    dataset['Marital_Status'] = dataset['Marital_Status'].map(ms_map)

    dataset.drop(['Dt_Customer'], axis=1, inplace=True)

    dataset['Income'] = dataset['Income'].fillna(dataset['Income'].median())

    columns_to_scale = list(dataset.columns)
    columns_to_scale.remove('ID')
    
    scaler = MinMaxScaler()
    dataset[columns_to_scale] = scaler.fit_transform(dataset[columns_to_scale])

    X = dataset.copy()
    
    if key == 'train':
        columns_to_drop = ['AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'Response', 'Target']
        for col in columns_to_drop:
            try:
                X.drop([col], axis=1, inplace=True)
            except:
                pass
        y = dataset['Target']
        return X, y
    else:
        return X

In [11]:
X_train, y_train = prepare(train, key='train')

In [12]:
X_test = prepare(test, key='test')

In [13]:
from sklearn.naive_bayes import GaussianNB

In [14]:
nb = GaussianNB()
nb.fit(X_train, y_train)

,"priors priors: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
,"var_smoothing var_smoothing: float, default=1e-9Portion of the largest variance of all features that is added tovariances for calculation stability... versionadded:: 0.20",1e-09


In [15]:
result = nb.predict(X_test)

In [16]:
result = pd.DataFrame(result, columns=['Target'])

In [17]:
total = pd.concat([X_test['ID'], result.astype('int64')], axis=1)
total

,ID,Target
0,286,0
1,6374,0
2,1160,0
3,701,1
4,4084,1
...,...,...
443,10678,0
444,5790,0
445,6606,1
446,1631,1


In [18]:
total.to_csv('result.csv', index=False)